In [2]:
# Cell 1 — Setup and load all metrics
import sys
from pathlib import Path
import pandas as pd
import plotly.express as px

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.models.train_linear import train_linear_model
from src.models.train_random_forest import train_random_forest
from src.models.train_xgboost import train_xgboost

In [3]:
# Cell 2 — Train all three models (or load saved metrics if already run)
linear_model, linear_metrics = train_linear_model()
rf_model, rf_metrics = train_random_forest()
xgb_model, xgb_metrics = train_xgboost()

Train: 1,311 rows (2021–2023)
Val:   458 rows (2024)
Test:  694 rows (2025 onward)
Linear Regression (Ridge): MAE=0.978s | RMSE=2.137s | MedianAE=0.696s
Saved model to C:\Users\siraj\OneDrive\Project Portfolio\f1-qualifying-lap-predictor\models\artefacts\linear_ridge.pkl
Saved metrics to C:\Users\siraj\OneDrive\Project Portfolio\f1-qualifying-lap-predictor\models\metrics\linear_ridge_metrics.csv
Train: 1,311 rows (2021–2023)
Val:   458 rows (2024)
Test:  694 rows (2025 onward)
Random Forest: MAE=0.895s | RMSE=2.187s | MedianAE=0.432s
Saved model to C:\Users\siraj\OneDrive\Project Portfolio\f1-qualifying-lap-predictor\models\artefacts\random_forest.pkl
Saved metrics to C:\Users\siraj\OneDrive\Project Portfolio\f1-qualifying-lap-predictor\models\metrics\random_forest_metrics.csv
Train: 1,311 rows (2021–2023)
Val:   458 rows (2024)
Test:  694 rows (2025 onward)
XGBoost (default params): MAE=1.071s | RMSE=2.526s | MedianAE=0.454s
Saved model to C:\Users\siraj\OneDrive\Project Portfolio\f1-

In [4]:
# Cell 3 — Load Baseline A's metric from Phase 5 for comparison
baseline_a_mae = pd.read_csv(PROJECT_ROOT / "models/metrics/baseline_best.csv")["MAE"].iloc[0]

comparison = pd.DataFrame([
    {"Model": "Baseline A (Driver Rolling)", "MAE": baseline_a_mae},
    {"Model": linear_metrics["Model"], "MAE": linear_metrics["MAE"]},
    {"Model": rf_metrics["Model"], "MAE": rf_metrics["MAE"]},
    {"Model": xgb_metrics["Model"], "MAE": xgb_metrics["MAE"]},
])

comparison

,Model,MAE
0,Baseline A (Driver Rolling),0.8353
1,Linear Regression (Ridge),0.9779
2,Random Forest,0.8948
3,XGBoost (default params),1.0713


In [ ]:
# Cell 4 — Visual comparison against the benchmark
fig = px.bar(
    comparison, x="Model", y="MAE",
    title="Model Comparison — MAE vs Baseline A Benchmark",
    labels={"MAE": "MAE (seconds)"},
    text="MAE",
    color="Model"
)
fig.add_hline(
    y=baseline_a_mae, line_dash="dash", line_color="red",
    annotation_text="Baseline A", annotation_position="top left"
)
fig.update_traces(textposition="outside")
fig.update_layout(showlegend=False)
fig.show()

In [6]:
# Cell 5 — Residual analysis for the best model (assume XGBoost wins)
from src.models.data_split import load_features, time_based_split
from src.models.train_utils import get_feature_columns, prepare_xy

df = load_features()
train, val, test = time_based_split(df)
feature_cols = get_feature_columns(df)
X_val, y_val = prepare_xy(val, feature_cols)

val = val.copy()
val["Prediction"] = xgb_model.predict(X_val)
val["Residual"] = val["DeltaToFastest_s"] - val["Prediction"]

fig = px.scatter(
    val, x="Prediction", y="Residual",
    title="XGBoost Residuals vs Predictions",
    labels={"Prediction": "Predicted Delta (s)", "Residual": "Residual (Actual - Predicted)"},
    opacity=0.5
)
fig.add_hline(y=0, line_dash="dash", line_color="white")
fig.show()

Train: 1,311 rows (2021–2023)
Val:   458 rows (2024)
Test:  694 rows (2025 onward)


In [7]:
# Cell 6 — Error breakdown by circuit for the winning model
from src.models.evaluate import evaluate_by_group

by_circuit = evaluate_by_group(val, "DeltaToFastest_s", "Prediction", "EventName")
by_circuit.head(10)

,EventName,MAE
4,Bahrain Grand Prix,5.940380
6,British Grand Prix,2.572146
5,Belgian Grand Prix,2.205963
19,Saudi Arabian Grand Prix,2.110885
10,Emilia Romagna Grand Prix,1.242645
20,Singapore Grand Prix,1.065536
1,Australian Grand Prix,0.886401
13,Japanese Grand Prix,0.796285
11,Hungarian Grand Prix,0.734777
14,Las Vegas Grand Prix,0.685554


In [8]:
# Cell 7 — Error breakdown by team
by_team = evaluate_by_group(val, "DeltaToFastest_s", "Prediction", "Team")
by_team

,Team,MAE
4,Kick Sauber,1.931373
9,Williams,1.422147
8,Red Bull Racing,1.381790
0,Alpine,1.363764
3,Haas F1 Team,1.137226
7,RB,0.774740
1,Aston Martin,0.743440
5,McLaren,0.713321
6,Mercedes,0.671374
2,Ferrari,0.589422
